# Prediksi Penyakit Jantung — Baseline SVM & Random Forest
**Dataset:** Heart Failure Prediction (Kaggle)

**Metode:** Support Vector Machine (SVM) dan Random Forest

**Pipeline:** Load Data => Cleaning => Pra-Pemrosesan Data => Baseline Model => Evaluasi => Visualisasi

---

## 1. Import Library
Import library yang digunakan dalam pra-pemrosesan data, evaluasi, dan permodelan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, roc_auc_score, roc_curve,confusion_matrix, classification_report)

---
## 2. Load Data
Dataset yang digunakan adalah **Heart Disease Dataset** yang berisi data klinis pasien untuk memprediksi ada/tidaknya penyakit jantung.

**Kolom-kolom utama dalam dataset:**
- `Age` — Usia pasien (tahun)
- `Sex` — Jenis kelamin (M/F)
- `ChestPainType` — Tipe nyeri dada (ATA, NAP, ASY, TA)
- `RestingBP` — Tekanan darah saat istirahat (mm Hg)
- `Cholesterol` — Kadar kolesterol serum (mg/dl)
- `FastingBS` — Gula darah puasa > 120 mg/dl (1=Ya, 0=Tidak)
- `RestingECG` — Hasil EKG istirahat
- `MaxHR` — Detak jantung maksimum yang dicapai
- `ExerciseAngina` — Angina akibat olahraga (Y/N)
- `Oldpeak` — Depresi ST yang diinduksi olahraga
- `ST_Slope` — Kemiringan segmen ST puncak olahraga

In [ ]:
df = pd.read_csv("D:\Tubes Machine Learning\heart.csv")
print(f"Shape awal: {df.shape}\n")
print(df.head())

---
## 3. Pembersihan Data (Data Cleaning)
Tahap ini bertujuan untuk memastikan data bebas dari duplikat, nilai tidak valid, dan nilai ekstrem sebelum masuk ke proses pemodelan.

In [ ]:
# Hapus duplikat
df.drop_duplicates(inplace=True)

#### Penghapusan Duplikat
Data duplikat dapat menyebabkan **data leakage** model belajar dari baris yang sama di training dan testing set. Ini akan membuat performa model terlihat lebih bagus dari kenyataannya, sehingga menyebabkan model hanya "menghafal" pola yang sama berulang kali dan dapat menyebabkan hasil evaluasi menjadi bias 

In [ ]:
# Nilai 0 tidak valid → ganti median
for col in ['Cholesterol', 'RestingBP']:
    median = df.loc[df[col] != 0, col].median()
    df[col] = df[col].replace(0, median)

#### Penanganan Nilai 0 Tidak Valid
Kolom `Cholesterol` dan `RestingBP` ditemukan memiliki nilai 0 yang tidak mungkin terjadi secara medis. Nilai 0 tersebut diganti dengan **nilai median** dari data yang valid karena median lebih robust terhadap outlier dibandingkan mean. Karena nilai tersebut merupakan anomali dalam sebuah dataset sehingga perlu ditangani sebelum masuk ke dalam model

In [ ]:
# Winsorize outlier (1%–99%)
num_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
for col in num_cols:
    df[col] = df[col].clip(df[col].quantile(0.01), df[col].quantile(0.99))

print(f"Shape setelah cleansing: {df.shape}")

#### Penanganan Outlier dengan Winsorizing
Winsorization adalah metode penanganan outlier yang membatasi nilai ekstrem ke batas persentil tertentu tanpa menghapus baris data. Digunakan batas persentil **1% dan 99%**, (misalnya membatasi pada 1% data terendah dan 1% data tertinggi). Jika ada data yang melewati batas tersebut, nilainya akan ditarik atau disamakan dengan batas yang sudah ditentukan

---
## Pra-Pemrosesan Data
Tahap ini mengubah data ke format yang dapat diproses oleh algoritma machine learning.

In [ ]:
# Encoding kategorikal —  One-Hot Encoding
df = pd.get_dummies(df,columns=['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope'],drop_first=True)

#### Metode One-Hot Encoding
One-Hot Encoding berfungsi untuk mengubah atribut kategori menjadi bentuk numerik agar bisa diproses oleh algoritma machine learning.

Machine learning model bekerja dengan angka, bukan teks. Fitur kategorikal seperti `Sex` (M/F) dan `ChestPainType` (ATA/NAP/ASY/TA) harus dikonversi ke format numerik. 

In [ ]:
# train split
X = df.drop(columns='HeartDisease')
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=43, stratify=y
)

#### trainsplit
Data dibagi menjadi **80% training dan 20% testing** dengan parameter `stratify=y` untuk memastikan proporsi kelas Normal dan Heart Disease tetap seimbang di kedua bagian.

- 80% data untuk training memastikan model memiliki cukup data untuk belajar pola.
- 20% untuk testing memastikan kita punya sampel evaluasi yang cukup representatif.

`stratify=y`
- Stratifikasi memastikan **proporsi kelas (0 dan 1) sama** di training dan testing set.
- Tanpa stratifikasi, secara kebetulan testing set bisa memiliki lebih banyak kelas 0 atau 1, membuat evaluasi menjadi bias.

`random_state=43`
- Untuk **reprodusibilitas** setiap kali kode dijalankan, pembagian data akan selalu sama, sehingga hasil eksperimen bisa dibandingkan secara konsisten.



In [ ]:
# Scaling — wajib untuk SVM
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

#### Feature Scaling - StandardScaler

**Scaling wajib untuk model SVM**
- SVM bekerja dengan mengukur jarak antar titik data (menggunakan kernel).
- Jika fitur memiliki skala berbeda (misalnya `Age` 20-80 vs `Cholesterol` 100-600), fitur dengan skala lebih besar akan mendominasi perhitungan jarak dan bias.
- `StandardScaler` mengubah setiap fitur sehingga memiliki **mean=0 dan standar deviasi =1**

**model Random Forest tidak memerlukan scaling**
- Random Forest berbasis **decision tree** yang hanya membandingkan nilai dalam satu fitur 
- Perbandingan ini tidak terpengaruh oleh skala absolut, sehingga scaling tidak diperlukan.

---
## Baseline Model
Kedua model dijalankan menggunakan parameter default tanpa proses tuning apapun, dengan tujuan untuk mendapatkan gambaran awal performa masing-masing model sebagai titik acuan sebelum dilakukan optimasi.

In [ ]:
# --- Baseline SVM ---
svm_base = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_base.fit(X_train_sc, y_train)
print("\nBaseline SVM selesai dilatih.")

#### Support Vector Machine (SVM)

**Cara Kerja:**  
SVM mencari **hyperplane** (bidang pemisah) yang memaksimalkan **margin** (jarak antara hyperplane dengan titik data terdekat dari setiap kelas, yang disebut support vectors).

`kernel='rbf'`  
- RBF (Radial Basis Function) adalah kernel default yang **paling serbaguna**.
- Kernel ini memetakan data ke dimensi yang lebih tinggi secara implisit, memungkinkan SVM menemukan batas keputusan yang **non-linear**.
- Data medis seperti dataset ini biasanya tidak dapat dipisahkan secara linear, sehingga kernel RBF lebih sesuai dibanding kernel linear.

`C=1.0`  
- Parameter `C` mengontrol trade-off antara **margin yang lebar** vs **kesalahan klasifikasi**.
- `C` kecil → margin lebar, lebih toleran terhadap misclassification (underfitting).
- `C` besar → margin sempit, berusaha mengklasifikasikan semua data dengan benar (overfitting).
- Untuk baseline, `C=1.0` adalah titik awal yang netral dan umum digunakan.

`gamma='scale'`   
- `gamma` mengontrol jangkauan pengaruh satu training sample.
- `gamma='scale'` otomatis menyesuaikan dengan jumlah fitur dan variasi data.
- Lebih robust dibanding nilai fixed karena adaptif terhadap dimensi dan skala data.

`probability=True`  
- Secara default SVM hanya menghasilkan **class label** (0 atau 1).
- `probability=True` mengaktifkan estimasi probabilitas menggunakan **Platt Scaling**, yang diperlukan untuk menghitung **ROC-AUC**.

In [ ]:
# --- Baseline Random Forest ---
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)
print("Baseline Random Forest selesai dilatih.")

#### Random Forest

**Cara Kerja:**  
Random Forest adalah algoritma machine learning yang bekerja dengan cara membangun banyak pohon keputusan (decision tree) sekaligus, lalu menggabungkan hasil prediksi dari semua pohon tersebut untuk menghasilkan keputusan akhir

`n_estimators=100` 
- 100 tree adalah nilai default yang memberikan **keseimbangan antara performa dan waktu komputasi**.
- Lebih banyak tree umumnya meningkatkan performa, tetapi dengan diminishing returns setelah ~100 tree.
- Untuk baseline, 100 tree sudah cukup representatif.

---
## Fungsi Evaluasi
Fungsi ini digunakan untuk mengevaluasi kedua model secara konsisten menggunakan metrik yang sama.



In [ ]:
def evaluate_baseline(model, X_te, y_te, label, params):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
 
    accuracy  = accuracy_score(y_te, y_pred)
    precision = precision_score(y_te, y_pred)
    recall    = recall_score(y_te, y_pred)
    f1        = f1_score(y_te, y_pred)
    auc       = roc_auc_score(y_te, y_prob)
 
    print("\n" + "=" * 45)
    print(f"  BASELINE {label}")
    print("=" * 45)
    for k, v in params.items():
        print(f"  {k:<12}: {v}")
    print("-" * 45)
    print(f"  Accuracy  : {accuracy:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {auc:.4f}")
    print("=" * 45)
    print(f"\nClassification Report — {label}:")
    print(classification_report(y_te, y_pred, target_names=['Normal', 'Heart Disease']))
 
    return y_pred, y_prob, accuracy, precision, recall, f1, auc

# Evaluasi SVM
y_pred_svm, y_prob_svm, acc_svm, pre_svm, rec_svm, f1_svm, auc_svm = evaluate_baseline(
    svm_base, X_test_sc, y_test,
    label="SVM",
    params={'Kernel': 'rbf (default)', 'C': '1.0 (default)', 'Gamma': 'scale (default)'}
)
 
# Evaluasi Random Forest
y_pred_rf, y_prob_rf, acc_rf, pre_rf, rec_rf, f1_rf, auc_rf = evaluate_baseline(
    rf_base, X_test, y_test,
    label="Random Forest",
    params={'n_estimators': '100 (default)', 'max_depth': 'None (default)', 'max_features': 'sqrt (default)'}
)

---
## 7. Visualisasi Hasil
Visualisasi perbandingan hasil prediksi kedua model baseline.

In [ ]:
# ── Visualisasi 1: Confusion Matrix berdampingan ─────────────────
fig1, axes1 = plt.subplots(1, 2, figsize=(12, 5))
fig1.suptitle('Confusion Matrix — Baseline SVM vs Random Forest', fontsize=13, fontweight='bold')
 
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d',
            cmap='Blues', ax=axes1[0],
            xticklabels=['Normal', 'Heart Disease'],
            yticklabels=['Normal', 'Heart Disease'])
axes1[0].set(title=f'Baseline SVM\nAcc={acc_svm:.3f} | AUC={auc_svm:.3f}', ylabel='Actual', xlabel='Predicted')
 
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d',
            cmap='Greens', ax=axes1[1],
            xticklabels=['Normal', 'Heart Disease'],
            yticklabels=['Normal', 'Heart Disease'])
axes1[1].set(title=f'Baseline Random Forest\nAcc={acc_rf:.3f} | AUC={auc_rf:.3f}', ylabel='Actual', xlabel='Predicted')
 
plt.tight_layout()
plt.savefig('baseline_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

#### Confusion Matrix
Menampilkan detail hasil prediksi kedua model berapa yang benar dan berapa yang salah diprediksi per kelas.

Menampilkan **distribusi prediksi benar dan salah** secara detail:  
- **True Positive (TP):** Pasien sakit, diprediksi sakit  
- **True Negative (TN):** Pasien sehat, diprediksi sehat 
- **False Positive (FP):** Pasien sehat, diprediksi sakit   
- **False Negative (FN):** Pasien sakit, diprediksi sehat  

In [ ]:
# ── Visualisasi 2: ROC Curve gabungan ────────────────────────────
plt.figure(figsize=(8, 6))
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)
fpr_rf,  tpr_rf,  _ = roc_curve(y_test, y_prob_rf)
 
plt.plot(fpr_svm, tpr_svm, color='steelblue', lw=2, label=f'Baseline SVM (AUC={auc_svm:.3f})')
plt.plot(fpr_rf,  tpr_rf,  color='seagreen',  lw=2, label=f'Baseline RF  (AUC={auc_rf:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.fill_between(fpr_svm, tpr_svm, alpha=0.05, color='steelblue')
plt.fill_between(fpr_rf,  tpr_rf,  alpha=0.05, color='seagreen')
plt.title('ROC Curve — Baseline SVM vs Random Forest')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.savefig('baseline_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

#### ROC Curve
Membandingkan kemampuan kedua model dalam membedakan kelas Normal dan Heart Disease di berbagai threshold keputusan. Model dengan kurva lebih mendekati sudut kiri atas dan AUC lebih tinggi dianggap lebih baik.

Menampilkan **trade-off antara True Positive Rate dan False Positive Rate** pada berbagai threshold:  
- Kurva yang mendekati sudut kiri atas = model sangat baik  
- Garis diagonal = model tidak lebih baik dari tebakan acak  
- **AUC (Area Under Curve):** Semakin mendekati 1.0, semakin baik 

In [ ]:
# ── Visualisasi 3: Bar Chart perbandingan metrik ─────────────────
plt.figure(figsize=(10, 6))
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
svm_scores   = [acc_svm, pre_svm, rec_svm, f1_svm, auc_svm]
rf_scores    = [acc_rf,  pre_rf,  rec_rf,  f1_rf,  auc_rf]
x            = np.arange(len(metric_names))
width        = 0.35
 
bars1 = plt.bar(x - width/2, svm_scores, width, label='Baseline SVM',
                color='steelblue', alpha=0.85)
bars2 = plt.bar(x + width/2, rf_scores,  width, label='Baseline RF',
                color='seagreen',  alpha=0.85)
 
plt.ylim(0, 1.15)
plt.xticks(x, metric_names)
plt.title('Perbandingan Metrik — Baseline SVM vs Random Forest')
plt.ylabel('Score')
plt.legend()
 
for bar in bars1:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
 
plt.tight_layout()
plt.savefig('baseline_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

#### Bar Chart Perbandingan Metrik
Membandingkan lima metrik evaluasi utama antara kedua model secara visual.

---
## Kesimpulan Baseline
Berdasarkan hasil eksperimen baseline yang telah dilakukan, kedua model sudah menunjukkan performa yang cukup baik meskipun hanya menggunakan parameter default tanpa optimasi apapun.

- **SVM** cenderung menghasilkan **Recall lebih tinggi** — artinya lebih banyak pasien sakit yang berhasil terdeteksi, yang merupakan metrik paling penting dalam konteks prediksi penyakit jantung.
- **Random Forest** cenderung menghasilkan **Precision lebih tinggi** — artinya prediksi sakit yang dikeluarkan model lebih bisa dipercaya.

Hasil baseline ini akan menjadi titik acuan untuk mengukur peningkatan performa setelah dilakukan proses **hyperparameter tuning** pada tahap berikutnya menggunakan RandomizedSearchCV.